In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

GPKG = "data/catastro_manzana.gpkg"
ENV_FILE = ".env"
OUT_CSV = "sources/imagenes/candidatas_mapillary.csv"

API = "https://graph.mapillary.com/images"
FIELDS = "id,captured_at,compass_angle,computed_compass_angle,geometry,computed_geometry,is_pano,camera_type,width,height,creator,thumb_original_url"
MARGIN = 30          # Meters around the block where we look for photos
MAX_ANGLE = 60       # A (non panoramic) photo looks at the block if its direction is within this angle of the block
MIN_YEAR = 2018      # Recent photos only: the model shows the building today

In [ ]:
# Celda 2
import csv
import math
import datetime
import requests
import shapely
import geopandas as gpd
from pyproj import Transformer

In [ ]:
# Celda 3
# Token from the .env file (line "MAPILLARY_TOKEN=MLY|..."):
token = ""
with open(ENV_FILE, encoding="utf-8") as f:
    for line in f:
        if line.startswith("MAPILLARY_TOKEN="):
            token = line.strip().split("=", 1)[1]
print("Token found:", token != "")

In [ ]:
# Celda 4
# Search box: the block plus MARGIN meters, in longitude / latitude:
parcels = gpd.read_file(GPKG, layer="parcelas")
block = parcels.geometry.union_all()
xmin, ymin, xmax, ymax = block.buffer(MARGIN).bounds
to_lonlat = Transformer.from_crs(25830, 4326, always_xy=True)
lon_min, lat_min = to_lonlat.transform(xmin, ymin)
lon_max, lat_max = to_lonlat.transform(xmax, ymax)
bbox = ",".join(str(round(v, 6)) for v in [lon_min, lat_min, lon_max, lat_max])
print("bbox:", bbox)

In [ ]:
# Celda 5
# Ask Mapillary for the images in the box (the API gives at most 2000 per call):
params = {"access_token": token, "fields": FIELDS, "bbox": bbox, "limit": 2000}
answer = requests.get(API, params=params, timeout=120)
answer.raise_for_status()
images = answer.json()["data"]
print(len(images), "images in the box")

In [ ]:
# Celda 6
# Compass bearing (0 = north, 90 = east) from point a to point b, in UTM meters:
def bearing(ax, ay, bx, by):
    return math.degrees(math.atan2(bx - ax, by - ay)) % 360

# Smallest difference between two bearings: (350, 10) -> 20
def angle_difference(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)

# Side of the block where the camera is, as a compass sector seen from the block center:
SECTORS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
def sector(angle):
    return SECTORS[int(((angle + 22.5) % 360) // 45)]

In [ ]:
# Celda 7
# For each image: date, position, distance to the block and whether it looks at the block:
to_utm = Transformer.from_crs(4326, 25830, always_xy=True)
center = block.centroid
rows = []
for image in images:
    geometry = image.get("computed_geometry") or image["geometry"]
    lon, lat = geometry["coordinates"]
    x, y = to_utm.transform(lon, lat)
    camera = shapely.Point(x, y)

    # Nearest point of the block, and the direction from the camera to it:
    nearest = shapely.ops.nearest_points(camera, block.boundary)[1]
    to_block = bearing(x, y, nearest.x, nearest.y)
    compass = image.get("computed_compass_angle", image.get("compass_angle"))
    if image.get("is_pano"):
        looks_at_block = True
    elif compass is None:
        looks_at_block = False
    else:
        looks_at_block = angle_difference(compass, to_block) <= MAX_ANGLE

    date = datetime.datetime.fromtimestamp(image["captured_at"] / 1000, datetime.UTC).date()
    rows.append({
        "id": image["id"],
        "date": str(date),
        "year": date.year,
        "pano": bool(image.get("is_pano")),
        "camera_type": image.get("camera_type", ""),
        "width": image.get("width", ""),
        "height": image.get("height", ""),
        "creator": image.get("creator", {}).get("username", ""),
        "x": round(x, 2),
        "y": round(y, 2),
        "dist_m": round(camera.distance(block), 1),
        "side": sector(bearing(center.x, center.y, x, y)),
        "compass": compass,
        "looks_at_block": looks_at_block,
        "url": image.get("thumb_original_url", ""),
        "page": "https://www.mapillary.com/app/?pKey=" + str(image["id"]),
    })
print(len(rows), "images described")

In [ ]:
# Celda 8
# Keep recent images that look at the block:
selected = [row for row in rows if row["year"] >= MIN_YEAR and row["looks_at_block"]]
print(len(selected), "recent images that look at the block (", sum(r["pano"] for r in selected), "panoramic )")

# How many per side of the block and per year:
sides = {}
years = {}
for row in selected:
    sides[row["side"]] = sides.get(row["side"], 0) + 1
    years[row["year"]] = years.get(row["year"], 0) + 1
print("per side:", dict(sorted(sides.items())))
print("per year:", dict(sorted(years.items())))

In [ ]:
# Celda 9
# Save the list of candidates (nothing is downloaded yet):
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(selected[0].keys()))
    writer.writeheader()
    writer.writerows(selected)
print("Saved:", OUT_CSV)

In [ ]:
# Celda 10
# Download the selected images (file name = Mapillary id), skipping the ones already on disk.
# The image links expire after a while, so this cell must run right after Celda 5.
import os
import time

DOWNLOAD_DIR = "sources/imagenes/mapillary"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
for row in selected:
    path = DOWNLOAD_DIR + "/" + str(row["id"]) + ".jpg"
    row["local_file"] = path
    if os.path.exists(path):
        continue
    answer = requests.get(row["url"], timeout=120)
    answer.raise_for_status()
    with open(path, "wb") as f:
        f.write(answer.content)
    time.sleep(0.5)

total_mb = sum(os.path.getsize(DOWNLOAD_DIR + "/" + name) for name in os.listdir(DOWNLOAD_DIR)) / 1e6
print(len(os.listdir(DOWNLOAD_DIR)), "files |", round(total_mb, 1), "MB")

In [ ]:
# Celda 11
# Attribution list (Mapillary images are CC BY-SA 4.0):
with open(DOWNLOAD_DIR + "/atribuciones.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["local_file", "id", "date", "creator", "license", "page"])
    writer.writeheader()
    for row in selected:
        writer.writerow({"local_file": row["local_file"], "id": row["id"], "date": row["date"],
                         "creator": row["creator"], "license": "CC BY-SA 4.0", "page": row["page"]})
print("Saved:", DOWNLOAD_DIR + "/atribuciones.csv")